# Laboratorium 5

**Imię i nazwisko:** Daniel Stefański  
**Nazwa ćwiczenia:** Transfer learning z MobileNetV2 dla klasyfikacji kwiatów

# Zadania

Proszę o wykorzystanie wbudowanego datasetu **oxford_flowers102** dostępnego w bibliotece 
**tensorflow_datasets**. Będziemy używali modelu **MobileNetV2** wstępnie wytrenowanego na zbiorze **ImageNet** jako bazowej sieci *CNN*. Dataset zawiera obrazy kwiatów 102 klas. Zbiór danych podzielony jest na część treningową, walidacyjną i testową. Klasy wzajemnie się wykluczają i nie nakładają na siebie.

Proszę o zrealizowanie następujących zadań:

### Zadanie.1

Uruchom trening z ustawieniami domyślnymi (faza 1: 5 epok, faza 2: 5 epok). Zapisz wykresy i opisz: czy model się przeuczył? Gdzie zaczyna się fine-tuning i jak wpłynął na accuracy i loss?

Dla ustawien domyslnych wykonano trening w dwoch fazach: faza 1 obejmowala **5** epok z zamrozona baza **MobileNetV2**, a faza 2 kolejne **5** epok po odblokowaniu czesci warstw i uruchomieniu fine-tuningu. Trening na CPU trwal okolo **177 s**.

W fazie 1 model szybko poprawial wyniki. Accuracy na zbiorze treningowym wzroslo od okolo **0.066** do **0.868**, natomiast validation accuracy od **0.283** do **0.752**. Jednoczesnie loss spadl z **4.485** do **0.677**, a validation loss z **3.712** do **1.075**. Oznacza to, ze model skutecznie nauczyl sie podstawowych cech obrazow jeszcze przed fine-tuningiem.

Fine-tuning zaczyna sie po zakonczeniu **5** epoki, czyli od **6** epoki treningu. Na wykresie jest to miejsce zaznaczone pionowa linia przerywana. Po rozpoczeciu fine-tuningu accuracy treningowe wyraznie spadlo, a loss chwilowo wzrosl. Jest to normalne zjawisko, poniewaz po odblokowaniu czesci warstw model musi ponownie dostroic wieksza liczbe parametrow.

Mimo spadku metryk treningowych, fine-tuning lekko poprawil wyniki walidacyjne. Validation accuracy wzroslo z okolo **0.752** do maksymalnie **0.760**, a validation loss obnizyl sie z **1.075** do minimum okolo **1.062**. Poprawa nie byla duza, ale widac, ze druga faza pomogla delikatnie lepiej dopasowac model do danych.

Model wykazuje oznaki lekkiego przeuczenia juz pod koniec fazy 1, poniewaz accuracy treningowe jest znacznie wyzsze niz walidacyjne, a loss treningowy jest duzo nizszy od validation loss. Nie jest to jednak bardzo silny overfitting, bo metryki walidacyjne nadal sie poprawiaja i nie dochodzi do gwaltownego pogorszenia wynikow na walidacji.

Wniosek: fine-tuning zaczyna sie od **6** epoki i w tym eksperymencie przyniosl niewielka, ale widoczna poprawe validation accuracy i validation loss. Model nie przeuczyl sie mocno, ale widoczna jest umiarkowana roznica miedzy wynikami treningowymi i walidacyjnymi.

Wykres dla zadania 1:

![Zadanie 1 - trening domyslny](task1_default_plot.png)


### Zadanie.2

Przeprowadź dwa eksperymenty:  
a. faza 1: 3 epoki, faza 2: 3 epoki  
b. faza 1: 10 epok, faza 2: 10 epok  
Porównaj wykresy i metryki (Precision, Recall, F1). Który wariant dał lepsze wyniki i 
dlaczego? 

W zadaniu porownano dwa warianty treningu: **3 + 3** epoki oraz **10 + 10** epok. Oba eksperymenty wykonano na tym samym modelu i tym samym zbiorze **oxford_flowers102**, aby sprawdzic, jak dluzszy trening wplywa na jakosc klasyfikacji.

Dla wariantu **3 + 3** uzyskano nastepujace metryki makro: **Precision = 0.708**, **Recall = 0.677**, **F1 = 0.670**. Najlepsze validation accuracy wynioslo okolo **0.680**, a minimalne validation loss okolo **1.610**. Wyniki sa poprawne, ale widac, ze model nie mial jeszcze wystarczajaco duzo czasu, aby dobrze dopasowac sie do bardziej zlozonego problemu 102 klas.

Dla wariantu **10 + 10** wyniki okazaly sie wyraznie lepsze. Otrzymano **Precision = 0.807**, **Recall = 0.781**, **F1 = 0.776**, najlepsze validation accuracy okolo **0.789** oraz minimalne validation loss okolo **0.818**. Oznacza to, ze dluzszy trening pozwolil modelowi lepiej wykorzystac zarowno faze uczenia klasyfikatora, jak i fine-tuning odblokowanej czesci sieci bazowej.

Lepszy okazal sie wariant **10 + 10**, poniewaz osiagnal wyzsze wartosci wszystkich trzech metryk oraz lepsze accuracy i nizszy validation loss. Dodatkowe epoki daly modelowi wiecej czasu na nauczenie sie reprezentacji cech, a fine-tuning skuteczniej poprawil dopasowanie do danych. Trzeba jednak pamietac, ze dluzszy trening oznacza tez znacznie wiekszy czas obliczen.

Wniosek: w tym eksperymencie wariant **10 + 10** dal najlepsze wyniki i byl skuteczniejszy od **3 + 3**, poniewaz model mial wiecej czasu na uczenie oraz dostrajanie wag podczas fine-tuningu.

Wykresy do zadania 2:

### Eksperyment a: faza 1 = 3, faza 2 = 3
![Zadanie 2 - eksperyment 3+3](screen/task2_exp_3_3_plot.png)

### Eksperyment b: faza 1 = 10, faza 2 = 10
![Zadanie 2 - eksperyment 10+10](screen/task2_exp_10_10_plot.png)


### Zadanie.3

Na podstawie Confusion Matrix wskaż które klasy kwiatów są najczęściej mylone ze sobą. Sformułuj hipotezę dlaczego model popełnia te konkretne błędy.

Na podstawie macierzy pomylek dla najlepszego wariantu z zadania 2 (**10 + 10** epok) widac, ze model najczesciej mylil nastepujace pary klas:

- **corn poppy** i **anthurium**
- **spring crocus** i **magnolia**
- **watercress** i **hibiscus**
- **azalea** i **peruvian lily**
- **ball moss** i **red ginger**

Najwieksza liczba pomylek dla pojedynczej pary wynosila **3**, a kolejne najczestsze pomylki pojawialy sie **2** razy. Oznacza to, ze model ogolnie radzil sobie dobrze, ale dla niektorych klas mial trudnosc z rozroznieniem subtelnych cech wizualnych.

Prawdopodobna przyczyna tych bledow jest podobienstwo wygladu kwiatow. Czesc klas ma zblizone kolory platkow, ksztalty kwiatostanow albo podobne tlo i sposob fotografowania. W przypadku zbioru 102 klas nawet niewielkie podobienstwa miedzy kwiatami moga powodowac pomylki, zwlaszcza gdy model widzi ograniczona liczbe przykladow dla kazdej klasy.

Druga mozliwa przyczyna to rozmiar i jakosc obrazow po przeskalowaniu do **160x160**. Przy takiej rozdzielczosci delikatne szczegoly odrozniajace klasy moga byc slabiej widoczne. Dodatkowo MobileNetV2 jest modelem lekkim, wiec mimo dobrej skutecznosci moze miec trudnosc z uchwyceniem bardzo drobnych roznic miedzy podobnymi kwiatami.

Wniosek: model najczesciej myli klasy, ktore sa do siebie wizualnie podobne pod wzgledem koloru, ksztaltu lub kompozycji zdjecia. To sugeruje, ze glownym zrodlem bledow nie jest przypadkowosc, lecz rzeczywiste podobienstwo miedzy klasami oraz ograniczona ilosc szczegolow dostepnych po przeskalowaniu obrazow.

Macierz pomylek do zadania 3:

![Zadanie 3 - Confusion Matrix](screen/task3_confusion_matrix.png)


### Zadanie.4

Znajdź po 2 zdjęcia każdej klasy (daisy, dandelion, roses, sunflowers, tulips) — łącznie 10 zdjęć. Przetestuj każde przez zakładkę "Klasyfikacja obrazu" i zapisz wyniki Top-5. Oblicz własnoręcznie accuracy na tych 10 obrazach.

Do zadania wykorzystano 10 przykladowych obrazow z folderu **Przykladowe obrazy do sprawdzenia poprawnosci klasyfikacji**, po **2** dla kazdej z klas: **daisy**, **dandelion**, **roses**, **sunflowers**, **tulips**. Predykcje wykonano modelem z **cnn.ipynb** dla zbioru **flower_photos** po treningu **5 + 5** epok. Model osiagnal na walidacji najlepsze **accuracy = 0.913**, a minimalny **val_loss = 0.268**.

Wyniki Top-5 dla poszczegolnych obrazow:

- **110147301_ad921e2828.jpg** (**tulips**): 1. **roses** **65.5%**, 2. **tulips** **34.0%**, 3. **sunflowers** **0.5%**, 4. **daisy** **0.1%**, 5. **dandelion** **0.0%**
- **118974357_0faa23cce9_n.jpg** (**roses**): 1. **roses** **99.7%**, 2. **tulips** **0.3%**, 3. **sunflowers** **0.0%**, 4. **daisy** **0.0%**, 5. **dandelion** **0.0%**
- **14957470_6a8c272a87_m.jpg** (**tulips**): 1. **tulips** **81.1%**, 2. **roses** **18.4%**, 3. **sunflowers** **0.5%**, 4. **daisy** **0.0%**, 5. **dandelion** **0.0%**
- **159079265_d77a9ac920_n.jpg** (**roses**): 1. **roses** **77.9%**, 2. **tulips** **22.0%**, 3. **daisy** **0.0%**, 4. **sunflowers** **0.0%**, 5. **dandelion** **0.0%**
- **25360380_1a881a5648.jpg** (**daisy**): 1. **daisy** **100.0%**, 2. **sunflowers** **0.0%**, 3. **roses** **0.0%**, 4. **dandelion** **0.0%**, 5. **tulips** **0.0%**
- **40410814_fba3837226_n.jpg** (**sunflowers**): 1. **sunflowers** **100.0%**, 2. **daisy** **0.0%**, 3. **tulips** **0.0%**, 4. **roses** **0.0%**, 5. **dandelion** **0.0%**
- **40411019_526f3fc8d9_m.jpg** (**sunflowers**): 1. **sunflowers** **98.7%**, 2. **daisy** **1.3%**, 3. **roses** **0.0%**, 4. **dandelion** **0.0%**, 5. **tulips** **0.0%**
- **5673551_01d1ea993e_n.jpg** (**daisy**): 1. **daisy** **87.7%**, 2. **sunflowers** **4.8%**, 3. **dandelion** **4.4%**, 4. **roses** **2.7%**, 5. **tulips** **0.3%**
- **8475769_3dea463364_m.jpg** (**dandelion**): 1. **sunflowers** **83.5%**, 2. **dandelion** **16.4%**, 3. **tulips** **0.1%**, 4. **roses** **0.0%**, 5. **daisy** **0.0%**
- **9818247_e2eac18894.jpg** (**dandelion**): 1. **daisy** **68.9%**, 2. **sunflowers** **25.5%**, 3. **tulips** **4.8%**, 4. **dandelion** **0.7%**, 5. **roses** **0.0%**

Poprawnie sklasyfikowanych zostalo **7** z **10** obrazow, dlatego recznie policzone **accuracy** wynosi:

**accuracy = 7 / 10 = 0.70 = 70%**

Najczestsze pomylki dotyczyly klas wizualnie podobnych. Jedno zdjecie **tulips** zostalo przewidziane jako **roses**, a dwa obrazy **dandelion** zostaly pomylone odpowiednio z **sunflowers** i **daisy**. Pokazuje to, ze model dobrze radzi sobie z rozpoznawaniem roznych typow kwiatow, ale nadal moze mylic klasy o podobnym kolorze lub ksztalcie platkow.

Wykres treningu modelu wykorzystanego do klasyfikacji:

![Zadanie 4 - trening modelu 5 klas](screen/task4_training_plot.png)


### Zadanie.5

Odpowiedz na pytania:  

a. Dlaczego zamrażamy bazę w fazie 1?  
b. Co daje fine-tuning w fazie 2?  
c. Jaką przewagę ma MobileNetV2 wytrenowany na ImageNet nad siecią trenowaną od zera na tym datasecie?

a. W fazie 1 zamrazamy baze **MobileNetV2**, aby zachowac wstepnie nauczone cechy z **ImageNet** i trenowac glownie nowa warstwe klasyfikacyjna. Dzi?ki temu uczenie jest stabilniejsze, szybsze i zmniejsza ryzyko zniszczenia uzytecznych wag juz na poczatku treningu. Jest to szczegolnie wazne wtedy, gdy nowy zbior danych jest mniejszy niz zbiory uzywane do trenowania duzych modeli od zera.

b. Fine-tuning w fazie 2 polega na odblokowaniu czesci warstw sieci bazowej i delikatnym dostrojeniu ich do nowego zadania. Pozwala to modelowi lepiej dopasowac reprezentacje cech do obrazow kwiatow, co zwykle poprawia **accuracy** i obniza **loss**. W praktyce faza ta pomaga przejsc od ogolnych cech obrazu do cech bardziej specyficznych dla rozwiazywanego problemu.

c. **MobileNetV2** wytrenowany na **ImageNet** ma przewage nad siecia trenowana od zera, poniewaz startuje juz z wagami, ktore nauczyly sie rozpoznawac krawedzie, tekstury, ksztalty i bardziej zlozone wzorce wizualne. Dzi?ki temu potrzeba mniej danych, mniej czasu treningu i zwykle latwiej osiagnac lepsze wyniki. Trenowanie od zera byloby wolniejsze, bardziej kosztowne obliczeniowo i bardziej podatne na slabsza generalizacje, zwlaszcza przy ograniczonej liczbie przykladow dla kazdej klasy.
